<a href="https://colab.research.google.com/github/MatthewFaiello/MLsessions/blob/main/ISEA_Week5_ML_HW.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Your Task: Predict Student Success

The purpose of this HW is to get you hands on with real data trying out the modelling techniques we talked about.

You are free to use gen-ai with this project to help with the coding (of course, you don't have to!). [Hands on Machine Learning](https://www.oreilly.com/library/view/hands-on-machine-learning/9781098125967/) is also a great resource.

Your code needs to run, but I want you to focus less on the specifics of the code and more on understanding the component steps that go into building and validating a model. Creating code is now pretty easy, creating a "good" model is hard.

For this exercise we will use open data on student dropout from Portugal. Full documentation is available [here](https://archive.ics.uci.edu/dataset/697/predict+students+dropout+and+academic+success)

M.V.Martins, D. Tolledo, J. Machado, L. M.T. Baptista, V.Realinho. (2021) "Early prediction of student’s performance in higher education: a case study" Trends and Applications in Information Systems and Technologies, vol.1, in Advances in Intelligent Systems and Computing series. Springer. DOI: 10.1007/978-3-030-72657-7_16

You will turn in on the class website a google slide deck that has:
1. A cover slide contianing your name (and all group member names if working together) and a link to your colab (**Create slide 1 now**)
2. 3 slides answering the questions below - they are clearly indicated as you go through the colab notebook.


# Get the data

Here I provide some code to get the data for you

In [119]:
!pip install ucimlrepo
from ucimlrepo import fetch_ucirepo

import pandas as pd

from sklearn.model_selection import train_test_split

import numpy as np

In [120]:
# fetch dataset
predict_students_dropout_and_academic_success = fetch_ucirepo(id=697)

# data (as pandas dataframes)
X = predict_students_dropout_and_academic_success.data.features
y = predict_students_dropout_and_academic_success.data.targets
df = df = X.join(y)

# metadata
print(predict_students_dropout_and_academic_success.metadata)

# variable information
print(predict_students_dropout_and_academic_success.variables)

{'uci_id': 697, 'name': "Predict Students' Dropout and Academic Success", 'repository_url': 'https://archive.ics.uci.edu/dataset/697/predict+students+dropout+and+academic+success', 'data_url': 'https://archive.ics.uci.edu/static/public/697/data.csv', 'abstract': "A dataset created from a higher education institution (acquired from several disjoint databases) related to students enrolled in different undergraduate degrees, such as agronomy, design, education, nursing, journalism, management, social service, and technologies.\nThe dataset includes information known at the time of student enrollment (academic path, demographics, and social-economic factors) and the students' academic performance at the end of the first and second semesters. \nThe data is used to build classification models to predict students' dropout and academic sucess. The problem is formulated as a three category classification task, in which there is a strong imbalance towards one of the classes.", 'area': 'Social Sc

# 1 Data Checking

- Look at your outcome variable - any cases to exclude?
- Determine the base-rate accuracy for a naive model
- Create Test and Training Sets
- Look at distributions of x variables, look up meta data, decide which to include

At the end of this section you should have
`x_train`, `x_text`, `y_train`, `y_test`
And an estimate of the base rate accuracy.

In [134]:
# check target distribution
pd.DataFrame({"count": df["Target"].value_counts(dropna=False),
              "percent": df["Target"].value_counts(normalize=True, dropna=False).mul(100).round(2)})

# recode so target is either "Graduate" or "Other"
df["Target_"] = df["Target"].replace({"Dropout": 0, "Enrolled": 0, "Graduate": 1})

# target base-rate
pd.DataFrame({"count": df["Target_"].value_counts(dropna=False),
              "percent": df["Target_"].value_counts(normalize=True, dropna=False).mul(100).round(2)})

# check features
metaData = predict_students_dropout_and_academic_success.variables; metaData
num = df.select_dtypes(include="number")
pd.DataFrame({"mean": num.mean(),
              "median": num.median(),
              "mode": num.mode(dropna=True).iloc[0]})

# select features
range_cols = df.iloc[:, np.r_[4, 6, 12:32, 37]].columns
df_subset = df.loc[:, range_cols]; df_subset

# stratified test and train sets
X = df_subset.drop(columns=["Target_"])
y = df_subset["Target_"]

# split (stratified)
X_train, X_test, y_train, y_test = train_test_split(X, y,
                                                    test_size=0.2,
                                                    random_state=42,
                                                    stratify=y)
# check stratification
pd.DataFrame({"y_train": y_train.value_counts(normalize=True, dropna=False).mul(100).round(2),
              "y_test": y_test.value_counts(normalize=True, dropna=False).mul(100).round(2)})

/tmp/ipython-input-303/1281890171.py:6: FutureWarning: Downcasting behavior in `replace` is deprecated and will be removed in a future version. To retain the old behavior, explicitly call `result.infer_objects(copy=False)`. To opt-in to the future behavior, set `pd.set_option('future.no_silent_downcasting', True)`
  df["Target_"] = df["Target"].replace({"Dropout": 0, "Enrolled": 0, "Graduate": 1})


,y_train,y_test
Target_,,
0,50.07,50.06
1,49.93,49.94


# 2 Train a Model
* Pick one of the models we discussed today and train it.
* Report its accuracy and print a confusion matrix.
   * How much better is your model than the base rate?
   * How does accuracy on the train set compare to accuracy on the test set?
   * **Report Slide 2: Include an image of the confusion matrix, the base rate accuracy, train-set accuracy and test-set accuracy**

In [213]:
# -----------------------------------------------------------------------------
# Logistic regression (L1) pipeline + grid search + evaluation + coefficient inspection
# -----------------------------------------------------------------------------
# This script:
#  1) builds a preprocessing+model pipeline (StandardScaler -> LogisticRegression w/ L1)
#  2) tunes hyperparameters via GridSearchCV using *balanced accuracy*
#  3) evaluates on train/test using BOTH accuracy and balanced accuracy
#  4) prints confusion matrix + classification report
#  5) inspects non-zero (selected) features + ranks by absolute coefficient magnitude
#  6) compares test accuracy to a majority-class baseline
# -----------------------------------------------------------------------------

from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import GridSearchCV, StratifiedKFold
from sklearn.metrics import (classification_report,
                             confusion_matrix,
                             balanced_accuracy_score,
                             accuracy_score)

# -----------------------------------------------------------------------------
# 1) Define the pipeline
# -----------------------------------------------------------------------------
# Pipeline ensures that scaling happens *inside* CV folds (prevents leakage)
# and is applied consistently at train/test time.
pipe = Pipeline(
    steps=[
        ("scaler", StandardScaler()),  # standardize features: mean=0, std=1
        ("clf", LogisticRegression(
            penalty="l1",       # L1 encourages sparsity (many coef = 0) => feature selection-ish
            solver="saga",      # required solver for L1 in sklearn LogisticRegression
            max_iter=10000,     # large to help convergence for L1 problems
            random_state=42
        )),
    ]
)

# -----------------------------------------------------------------------------
# 2) Hyperparameter grid
# -----------------------------------------------------------------------------
# C is inverse regularization strength:
#   - smaller C => stronger regularization => more coefficients pushed to zero
#   - larger C => weaker regularization => more complex model
# class_weight:
#   - None: no reweighting
#   - "balanced": reweights classes inversely proportional to their frequency
param_grid = {"clf__C": [0.001, 0.01, 0.1, 1, 10, 100],
              "clf__class_weight": [None, "balanced"]}

# -----------------------------------------------------------------------------
# 3) Cross-validation strategy
# -----------------------------------------------------------------------------
# StratifiedKFold keeps class proportions similar in each fold, which is important
# for imbalanced classification problems.
cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)

# -----------------------------------------------------------------------------
# 4) Grid search configuration
# -----------------------------------------------------------------------------
# Scoring is *balanced_accuracy*: average recall across classes, less sensitive to imbalance.
# n_jobs=-1 uses all CPU cores.
gs = GridSearchCV(estimator=pipe,
                  param_grid=param_grid,
                  cv=cv,
                  scoring="balanced_accuracy",
                  n_jobs=-1,
                  refit=True)  # after CV, refit best params on full training set

# Fit grid search on training data only (good hygiene)
gs.fit(X_train, y_train)

print("Best params:", gs.best_params_)
print("CV best balanced acc:", gs.best_score_)

# -----------------------------------------------------------------------------
# 4b) Report CV variability for the best params (mean +/- std across folds)
# -----------------------------------------------------------------------------
# This helps contextualize best_score_ (how stable is it across folds?).
best_idx = gs.best_index_
cv_mean = gs.cv_results_["mean_test_score"][best_idx]
cv_std  = gs.cv_results_["std_test_score"][best_idx]
print(f"CV balanced acc (mean ± std): {cv_mean:.4f} ± {cv_std:.4f}")

# Best refit model: a *Pipeline* with scaler+classifier using the chosen params
best_model = gs.best_estimator_

# -----------------------------------------------------------------------------
# 5) Train/test evaluation: accuracy vs balanced accuracy
# -----------------------------------------------------------------------------
# NOTE: .score() on a classifier returns *accuracy* by default.
# We compute both metrics explicitly to avoid confusion.
pred_train = best_model.predict(X_train)
pred_test  = best_model.predict(X_test)

train_accuracy = accuracy_score(y_train, pred_train)
test_accuracy  = accuracy_score(y_test, pred_test)

train_bal_acc = balanced_accuracy_score(y_train, pred_train)
test_bal_acc  = balanced_accuracy_score(y_test, pred_test)

print(f"\nTrain accuracy          : {train_accuracy:.4f}")
print(f"Train balanced accuracy : {train_bal_acc:.4f}")
print(f"Test accuracy           : {test_accuracy:.4f}")
print(f"Test balanced accuracy  : {test_bal_acc:.4f}")

# -----------------------------------------------------------------------------
# 6) Confusion matrix + classification report (test)
# -----------------------------------------------------------------------------
# Use classifier's learned class order to keep row/col labels consistent.
labels = best_model.named_steps["clf"].classes_

cm = confusion_matrix(y_test, pred_test, labels=labels)
print("\nConfusion matrix (test):")
print(
    pd.DataFrame(
        cm,
        index=[f"true_{l}" for l in labels],
        columns=[f"pred_{l}" for l in labels],
    )
)

print("\nClassification report (test):")
print(classification_report(y_test, pred_test))

# -----------------------------------------------------------------------------
# 7) Coefficient inspection / feature selection
# -----------------------------------------------------------------------------
# With L1 regularization, many coefficients become exactly zero.
# Non-zero coefficients correspond to "selected" features.
clf = best_model.named_steps["clf"]

# Feature names:
#  - If X_train is a pandas DataFrame, we can pull columns.
#  - If X_train is a numpy array, you must supply names yourself.
feature_names = getattr(X_train, "columns", None)
if feature_names is None:
    raise ValueError(
        "X_train has no .columns attribute. Provide feature_names manually "
        "or pass a pandas DataFrame into the pipeline."
    )

coefs = clf.coef_  # shape: (1, n_features) for binary; (n_classes, n_features) for multiclass

# Identify selected (non-zero) features:
#  - binary: coef_ is a single row
#  - multiclass: mark a feature selected if ANY class uses it (non-zero in any row)
if coefs.shape[0] == 1:
    selected_idx = np.where(coefs.ravel() != 0)[0]
else:
    selected_idx = np.where(np.any(coefs != 0, axis=0))[0]

selected_features = feature_names[selected_idx]
print(f"\nSelected features ({len(selected_features)}):")
print(list(selected_features))

# Rank features by absolute coefficient magnitude:
#  - binary: abs of the single coefficient vector
#  - multiclass: use max abs coefficient across classes as a simple global importance proxy
if coefs.shape[0] == 1:
    abs_w = np.abs(coefs.ravel())
else:
    abs_w = np.max(np.abs(coefs), axis=0)

ranking = np.argsort(-abs_w)  # descending order
top = pd.DataFrame({"feature": feature_names[ranking], "abs_weight": coefs.ravel()[ranking]})
print("\nFeatures by |coefficient|:")
print(top)

# -----------------------------------------------------------------------------
# 8) Compare test accuracy to a majority-class baseline (sanity check)
# -----------------------------------------------------------------------------
# Majority baseline = accuracy you get by predicting the most frequent class always.
# Helpful for understanding whether accuracy is meaningfully better than trivial.
if hasattr(y_test, "value_counts"):
    majority_acc = y_test.value_counts(normalize=True).max()
else:
    # fallback for numpy arrays (works best if labels are hashable)
    vals, counts = np.unique(y_test, return_counts=True)
    majority_acc = counts.max() / counts.sum()

improvement_pp = (test_accuracy - majority_acc) * 100

print(f"\nTest accuracy: {test_accuracy*100:.2f}%")
print(f"Majority baseline: {majority_acc*100:.2f}%")
print(f"Improvement: {improvement_pp:.2f} percentage points")


Classification report (test):
              precision    recall  f1-score   support

           0       0.86      0.77      0.82       443
           1       0.80      0.88      0.83       442

    accuracy                           0.83       885
   macro avg       0.83      0.83      0.83       885
weighted avg       0.83      0.83      0.83       885



# 3 Train a Different Model
* Repeat all the steps in 2, but use a different model
* In addition, compare the accuracy of 1 and 2
* **Report Slide 3: Model 2 confusion matrix, train-set accuracy and test-set accuracy. Comparison Model 1 and Model 2 accuracy**

In [ ]:
# -----------------------------------------------------------------------------
# Decision Tree pipeline + grid search + evaluation + feature importance
# -----------------------------------------------------------------------------

from sklearn.pipeline import Pipeline
from sklearn.tree import DecisionTreeClassifier

# -----------------------------------------------------------------------------
# 1) Define the pipeline
# -----------------------------------------------------------------------------
# Trees don't need scaling, so we just use the classifier.
pipe = Pipeline(
    steps=[
        ("clf", DecisionTreeClassifier(random_state=42))
    ]
)

# -----------------------------------------------------------------------------
# 2) Hyperparameter grid
# -----------------------------------------------------------------------------
# max_depth: limits tree depth (controls overfitting)
# min_samples_leaf: minimum samples in a leaf (regularization)
# min_samples_split: minimum samples to split a node
# criterion: split quality measure
# ccp_alpha: post-pruning strength (higher => more pruning)
param_grid = {
    "clf__criterion": ["gini", "entropy", "log_loss"],
    "clf__max_depth": [None, 3, 5, 10, 20],
    "clf__min_samples_split": [2, 5, 10, 20],
    "clf__min_samples_leaf": [1, 2, 5, 10],
    "clf__ccp_alpha": [0.0, 0.001, 0.01],
    "clf__class_weight": [None, "balanced"],
}

# -----------------------------------------------------------------------------
# 3) Cross-validation strategy
# -----------------------------------------------------------------------------
cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)

# -----------------------------------------------------------------------------
# 4) Grid search configuration
# -----------------------------------------------------------------------------
gs = GridSearchCV(
    estimator=pipe,
    param_grid=param_grid,
    cv=cv,
    scoring="balanced_accuracy",
    n_jobs=-1,
    refit=True
)

gs.fit(X_train, y_train)

print("Best params:", gs.best_params_)
print("CV best balanced acc:", gs.best_score_)

# -----------------------------------------------------------------------------
# 4b) Report CV variability for the best params (mean +/- std across folds)
# -----------------------------------------------------------------------------
best_idx = gs.best_index_
cv_mean = gs.cv_results_["mean_test_score"][best_idx]
cv_std  = gs.cv_results_["std_test_score"][best_idx]
print(f"CV balanced acc (mean ± std): {cv_mean:.4f} ± {cv_std:.4f}")

best_model = gs.best_estimator_

# -----------------------------------------------------------------------------
# 5) Train/test evaluation: accuracy vs balanced accuracy
# -----------------------------------------------------------------------------
pred_train = best_model.predict(X_train)
pred_test  = best_model.predict(X_test)

train_accuracy = accuracy_score(y_train, pred_train)
test_accuracy  = accuracy_score(y_test, pred_test)

train_bal_acc = balanced_accuracy_score(y_train, pred_train)
test_bal_acc  = balanced_accuracy_score(y_test, pred_test)

print(f"\nTrain accuracy          : {train_accuracy:.4f}")
print(f"Train balanced accuracy : {train_bal_acc:.4f}")
print(f"Test accuracy           : {test_accuracy:.4f}")
print(f"Test balanced accuracy  : {test_bal_acc:.4f}")

# -----------------------------------------------------------------------------
# 6) Confusion matrix + classification report (test)
# -----------------------------------------------------------------------------
labels = best_model.named_steps["clf"].classes_

cm = confusion_matrix(y_test, pred_test, labels=labels)
print("\nConfusion matrix (test):")
print(
    pd.DataFrame(
        cm,
        index=[f"true_{l}" for l in labels],
        columns=[f"pred_{l}" for l in labels],
    )
)

print("\nClassification report (test):")
print(classification_report(y_test, pred_test))

# -----------------------------------------------------------------------------
# 7) Feature importance inspection (tree equivalent of coefficients)
# -----------------------------------------------------------------------------
clf = best_model.named_steps["clf"]

feature_names = getattr(X_train, "columns", None)
if feature_names is None:
    feature_names = np.array([f"x{i}" for i in range(X_train.shape[1])])
else:
    feature_names = np.array(feature_names)

importances = clf.feature_importances_
ranking = np.argsort(-importances)

top = pd.DataFrame(
    {"feature": feature_names[ranking], "importance": importances[ranking]}
)

# Often many are 0.0 if the feature never used in splits
print("\nTop features by importance:")
print(top.head(30))

# -----------------------------------------------------------------------------
# 8) Compare test accuracy to a majority-class baseline (sanity check)
# -----------------------------------------------------------------------------
if hasattr(y_test, "value_counts"):
    majority_acc = y_test.value_counts(normalize=True).max()
else:
    vals, counts = np.unique(y_test, return_counts=True)
    majority_acc = counts.max() / counts.sum()

improvement_pp = (test_accuracy - majority_acc) * 100

print(f"\nTest accuracy: {test_accuracy*100:.2f}%")
print(f"Majority baseline: {majority_acc*100:.2f}%")
print(f"Improvement: {improvement_pp:.2f} percentage points")

# 4 Reflection
* **Type responses on Slide 4**
* Contextualizing accuracy - think about different use cases for your model, which ones would you feel its accurate enough to use for? I only asked you to look at overall accuracy, is that good enough?
* Contextualizing features - think about these same use cases, are the prediction features you included appropriate for these uses?
* Generalizability - again thinking about your features, could you use this model in other educational contexts? How hard would it be to get that same data? Are there issues with it generalizing over time and location?

# 5 Extra Credit
* Consider ensembling your two models. Does that perform better?
* Check accuracy for different subgroups